# Pipeline v13.6 -- Qwen BoN-SC (Option B) + Rich Calibrator Features (Option C)
**EXACT 2026 -- XAI Challenge @ IJCNN | Kaggle T4x2 | best so far: v13.5 = 58.0% FULL**

## Hai cai tien gop chung (1 rebuild)

### Option B -- Qwen-CoT Best-of-N + Self-Consistency
- Stage D: moi cau sinh N=5 CoT (temp=0.5), vote tren Final Answer
- `pure_qwen` gio = majority vote, on dinh hon n=1
- Sinh ra confidence signal: `qwen_sc_conf` (ti le vote)

### Option C -- Features moi cho calibrator
- `qwen_sc_conf`     : do dong thuan BoN cua Qwen (confidence that su)
- `qwen_vote_spread` : so dap an phan biet trong N=5
- `qwen_n_cited`     : so premise Qwen trich dan
- `z3_qwen_jaccard`  : **semantic agreement** -- Jaccard giua Z3 unsat-core
                       va premise Qwen trich dan (do hai he "cung ly do" khong)

## Decider moi
`CONF_HYBRID`: khi Z3 va Qwen bat dong, tin Qwen CHI khi qwen_sc_conf >= 0.6
va > z3_conf; nguoc lai giu Z3.

## Du kien
- pure_qwen: 50.1% -> ~52-54% (SC vote)
- calibrator: 58.0% -> ~59-61% (features tot hon)
- conf_hybrid: moi, ky vong ~58-60%
- ORACLE: 65.6% (gan nhu khong doi -- gioi han model)

## Config
```python
QWEN_BON_N=5  QWEN_BON_TEMP=0.5  QWEN_CONF_TRUST=0.6
FORCE_REBUILD=True   # doi False sau lan dau
FEATURES_CACHE_PATH="/kaggle/working/pipeline_features_v13_6.json"
```

## Runtime
~70-85 phut (Qwen BoN N=5 la phan chinh). Trong gioi han 12h.


In [ ]:
#!/usr/bin/env python3
"""
notebook_v13_4_with_lora.py -- EXACT 2026 v13.4
Same architecture as v13.3 (full comparison) but Stage D (Qwen-CoT)
uses LoRA v14 trained on dataset's `explanation` field.
"""


In [ ]:
# Fix Kaggle T4
import os
os.environ['VLLM_USE_V1'] = '0'


In [ ]:

# ==================================================================
# STAGE 0 -- Install Dependencies & Config
# ==================================================================

import subprocess, sys

pkgs = [
    "vllm",
    "z3-solver",
]
print("Installing vLLM + z3-solver (co the mat 2-5 phut)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages"]
    + pkgs,
    check=True,
)
print("All packages installed OK")

import json, os, re, time, csv, traceback
from pathlib import Path
from dataclasses import dataclass, field


# --- v8.1 FIX: KILL FLASHINFER AFTER PIP INSTALL ---
import os, sys, shutil, glob
for d in glob.glob("/usr/local/lib/python3.12/dist-packages/flashinfer*"):
    tgt = d + "_DISABLED"
    if os.path.exists(d) and not os.path.exists(tgt):
        try: os.rename(d, tgt)
        except: shutil.rmtree(d, ignore_errors=True)
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
for mod in list(sys.modules.keys()):
    if "flashinfer" in mod: del sys.modules[mod]
os.environ["VLLM_ATTENTION_BACKEND"] = "FLASH_ATTN"
# ---------------------------------------------------

import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from z3 import (
    Int, IntSort, IntVal, BoolSort, Function,
    ForAll, Exists, And, Or, Not, Implies, Solver, sat, unsat,
)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}    : {props.name}  ({props.total_memory / 1024**3:.1f} GB)")
print("Imports OK")
import subprocess, sys
try:
    import lightgbm
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","--quiet","--break-system-packages","lightgbm"], check=False)
print("lightgbm install attempted")


In [ ]:

# ---- Robust Kaggle path resolver (handles mount-path variations) ----
import os as _os
def _resolve(cands, label="path"):
    for p in cands:
        if _os.path.exists(p):
            print(f"  resolved {label}: {p}")
            return p
    print(f"  WARNING {label}: none of candidates exist; using first: {cands[0]}")
    return cands[0]

TRAIN_PATH = _resolve([
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
], "TRAIN")
TEST_PATH = _resolve([
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
], "TEST")
DATASET_PATH = TRAIN_PATH    # eval notebooks use the labeled train file


# ==================================================================
# CAU HINH -- Chinh sua o day
# ==================================================================
QWEN_MODEL_ID  = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
# DATASET_PATH set by resolver
PHYSICS_# DATASET_PATH set by resolver  # v8.2: Skip physics, focus on logic

# --- Data Split Config ---
SEED = 42
TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
RUN_ON_SPLIT = "all"  # v8.2: Full dataset  # 'test', 'train', 'val', hoac 'all'

N_SAMPLES      = 411
# --- Best-of-N Config ---
BEST_OF_N       = 3  # v8.2: Reduced for time budget     # Number of candidates per sample
BON_TEMPERATURE = 0.5  # v8.2: Lower for consistency   # Higher temp for diversity

# --- v9: Optional CoT for Formalization (Pass 1) ---
# False = giu nguyen baseline v8.2 (da kiem chung 50.0%).
# True  = cho LLM "thinking" truoc khi chot JSON AST -> ky vong giam no_ast, danh thuc Z3.
FORMALIZE_WITH_COT = False

MAX_RETRIES    = 2               # Giam xuong 2 vi v4 nhe hon
OUTPUT_PATH    = "/kaggle/working/pipeline_results_v12_1.json"
MAX_NEW_TOKENS_PASS1 = 4096      # Token cho Premises
MAX_NEW_TOKENS_PASS2 = 1024      # Token cho Question (it hon)
ANS_MAX_TOKENS       = 600       # Token cho Qwen Fallback

# vLLM Config
TENSOR_PARALLEL  = min(N_GPUS, 2)
MAX_MODEL_LEN    = 8192
GPU_MEMORY_UTIL  = 0.85
DTYPE            = "half"

# Physics Config
PHYSICS_MODE       = "direct"
PHYSICS_MAX_TOKENS = 1024
PHYSICS_TOLERANCE  = 0.05

# Z3 Entailment Config
Z3_ENTAILMENT      = True
Z3_SOLVER_TIMEOUT  = 5000
Z3_SELF_CORRECT    = False   # v10: OFF (24/114 resolved, 9 -> Unknown; low ROI)
Z3_SC_MAX_RETRIES  = 1       # Max self-correction rounds
REPETITION_PENALTY = 1.1     # v8.1 from exact.ipynb: chong token loop
# ==================================================================

# ==================================================================
# v10 -- Z3 FIDELITY CONFIG
# ==================================================================
# Quality Gate: chi cho Z3 tra loi khi formalization dang tin cay.
Z3_QUALITY_GATE          = True
GATE_REQUIRE_FULL_COMPILE = True   # compiled_count == total_count
GATE_REJECT_HALLUCINATION = True   # 0 hallucination warning
# Neu gate fail -> cau hoi do chuyen sang Qwen CoT fallback.

# Formalizer-LoRA (STaR loop). De rong "" = dung model goc cho moi pass.
FORMALIZER_LORA_PATH = ""          # vd: "/kaggle/working/qwen3_formalizer_lora"
LORA_MAX_RANK        = 16

# Export verified formalizations (chay tren TRAIN de tao data train cho LoRA)
EXPORT_VERIFIED        = False
VERIFIED_OUTPUT_PATH   = "/kaggle/working/verified_formalizations.json"

# ==================================================================
# v11 -- GOLD-FOL CONFIG
# ==================================================================
PREMISE_SOURCE   = "gold"     # "gold" (parse premises-FOL) | "lora_fol" (NL->FOL via LoRA)
USE_IDX_FILTER   = False      # True: chi dung premise trong idx[q] (gold supporting set)
FOL_FALLBACK_TO_QWEN = True   # mau parse FOL that bai -> Qwen CoT truc tiep

# ==================================================================
# v12 -- BoN + SELF-CONSISTENCY + GATING + UNSAT-CORE
# ==================================================================
BON_N_QFORMALIZE     = 5           # candidates per Pass-2 question
BON_TEMP             = 0.7         # generation diversity
SC_CONFIDENCE_TAU    = 0.6         # Tier-1 vote-share threshold
GROUNDED_FRAC_TAU    = 0.6         # Tier-1 grounded-candidates threshold
EXTRACT_UNSAT_CORE   = True        # produce support_idx for Z3=Yes

# v12.1 -- post-mortem fixes
USE_Z3_INFORMED_FALLBACK = True   # Qwen fallback sees premise FOL + Z3 votes + Z3 verdict
                                  # (v12 showed qwen_fallback @ 36%; this aims for 45-50%)
MAX_PASS2_TOKENS_YN  = 200
MAX_PASS2_TOKENS_MC  = 500

# v12 overrides v11 defaults:
USE_IDX_FILTER       = True        # ON by default in v12 (was False in v11)

print("Config OK - v12.1 (Z3-informed fallback + Qwen-priority arbiter)")

# ==================================================================
# v13 -- ARCHITECTURE COMPARISON
# ==================================================================
CAL_SEED            = 42
CAL_TRAIN_RATIO     = 0.80
CAL_TAU             = 0.55         # overridden by tuned tau in v13.1/v13.3
PH_HIGH_CONF        = 0.7          # Parallel-Hybrid threshold: trust Z3 on disagreement
FEATURES_CACHE_PATH = "/kaggle/working/pipeline_features_v136_v4.json"
LOAD_FEATURES_CACHE = True         # use cache if exists; else build & save
FORCE_REBUILD = True        # set True to ignore cache

print("v13 config extras loaded")

# ==================================================================
# v13.4 -- LoRA v14 for Qwen-CoT step
# ==================================================================
# Path to v14 LoRA adapter (PEFT format). Upload from previous Kaggle output.
COT_LORA_PATH = "/kaggle/input/notebooks/nguyenminhtric/v14-fine-tune/qwen3_cot_lora_v14_v4"
USE_COT_LORA  = True

# Override v13 cache so we don\'t collide with base-Qwen features
FEATURES_CACHE_PATH = "/kaggle/working/pipeline_features_v136_v4.json"

print("v13.4 config:")
print(f"  COT_LORA_PATH       = {COT_LORA_PATH}")
print(f"  USE_COT_LORA        = {USE_COT_LORA}")
print(f"  FEATURES_CACHE_PATH = {FEATURES_CACHE_PATH}")

# ==================================================================
# v13.5 -- System prompt fix (CRITICAL: must match v14 training prompt)
# ==================================================================
# This EXACT prompt was used to train LoRA v14. Must be identical at inference.
V14_COT_SYSTEM = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'). "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)
COT_MAX_TOKENS = 768    # explanations avg ~85 tok; 768 gives ample room

# New cache (old v14 cache has wrong-prompt qwen answers, must rebuild)
FEATURES_CACHE_PATH = "/kaggle/working/pipeline_features_v136_v4.json"
FORCE_REBUILD = True    # set False after first run

print("v13.5 config: V14_COT_SYSTEM defined, cache -> pipeline_features_v13_5.json")

# ==================================================================
# v13.6 -- Option B (Qwen-CoT BoN+SC) + Option C (richer calibrator features)
# ==================================================================
QWEN_BON_N      = 5       # Qwen-CoT self-consistency samples per question
QWEN_BON_TEMP   = 0.5     # diversity for SC voting
QWEN_CONF_TRUST = 0.6     # conf_hybrid: trust Qwen on disagreement if SC-conf >= this

FEATURES_CACHE_PATH = "/kaggle/working/pipeline_features_v136_v4.json"
FORCE_REBUILD = True      # set False after first run

print("v13.6 config: Qwen BoN-SC + richer features, cache -> pipeline_features_v13_6.json")


In [ ]:

# ==================================================================
# STAGE 1 -- Load vLLM Engine
# ==================================================================

print(f"\nLoading vLLM engine ({QWEN_MODEL_ID})...")
print("  (Lan dau tai ~15 GB tu HuggingFace, sau do cache)")

t_load = time.time()

_USE_LORA = bool(FORMALIZER_LORA_PATH) or bool(COT_LORA_PATH and USE_COT_LORA)
llm = LLM(
    model=QWEN_MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL,
    dtype=DTYPE,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    trust_remote_code=True,
    enforce_eager=True,        # v8.1 FIX: Disable CUDA graph to save 1.67 GiB VRAM on T4
    enable_lora=_USE_LORA,     # v10: serve formalizer LoRA if provided
    max_lora_rank=LORA_MAX_RANK,
)

# v10: build LoRA request (used ONLY for formalization passes)
LORA_REQ = None
# v13.4 fix: guard specifically on FORMALIZER_LORA_PATH (not _USE_LORA,
# which can be True just because COT_LORA_PATH is set)
if FORMALIZER_LORA_PATH:
    from vllm.lora.request import LoRARequest
    LORA_REQ = LoRARequest("formalizer", 1, FORMALIZER_LORA_PATH)
    print(f"Formalizer-LoRA enabled: {FORMALIZER_LORA_PATH}")
else:
    print("No formalizer LoRA (base model for all passes)")

tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_ID, trust_remote_code=True
)

t_loaded = time.time() - t_load
print(f"vLLM Engine loaded in {t_loaded:.1f}s")
print("OK")

# ==================================================================
# v13.4 -- Build CoT LoRA request (separate lora_int_id from formalizer)
# ==================================================================
COT_LORA_REQ = None
if COT_LORA_PATH and USE_COT_LORA:
    import os
    if os.path.exists(COT_LORA_PATH):
        from vllm.lora.request import LoRARequest
        COT_LORA_REQ = LoRARequest("cot_v14", 2, COT_LORA_PATH)
        print(f"CoT-LoRA enabled: {COT_LORA_PATH}")
    else:
        print(f"WARNING: COT_LORA_PATH not found at {COT_LORA_PATH}; falling back to base Qwen for CoT")
        USE_COT_LORA = False
else:
    print("CoT-LoRA disabled (base model for Qwen-CoT step)")


In [ ]:

# ==================================================================
# STAGE 2 -- Ontology & Prompts (Two-Pass)
# ==================================================================

GLOBAL_ONTOLOGY_TEXT = """
## GLOBAL ONTOLOGY -- BAT BUOC TUAN THU

### Quantifiers:
  forall -> forall  |  exists -> exists

### Logical Operators:
  and, or, implies, iff, not

### AST Node Types (4 loai):
  quantifier : { "type":"quantifier",  "operator":"forall|exists",
                 "bound_variables":["x",...], "body":{...} }
  connective : { "type":"connective",  "operator":"and|or|implies|iff|not",
                 "operands":[{...},...] }
  predicate  : { "type":"predicate",   "name":"PredicateName",
                 "arguments":["x","y",...] }
  variable   : { "type":"variable",    "name":"x" }
  constant   : { "type":"constant",    "name":"SomeName" }

### QUY TAC:
  1. Chi dung 4 node types tren
  2. 'not' chi co DUNG 1 operand
  3. 'implies' co DUNG 2 operands
  4. bound_variables phai la list
  5. Variables: lowercase (x,y,z), Constants: PascalCase
"""

# ------------------------------------------------------------------
# FEW-SHOT EXAMPLES (CRITICAL FOR QWEN-7B)
# ------------------------------------------------------------------

FEW_SHOT_PREMISES = """
### VÍ DỤ HOÀN CHỈNH:

Premises:
  "All students who study hard pass the exam."
  "Alice is a student."
  "Alice studies hard."

Output:
{
  "step1_local_ontology": [
    {"predicate": "Student",   "arity": 1, "description": "x is a student"},
    {"predicate": "StudyHard", "arity": 1, "description": "x studies hard"},
    {"predicate": "PassExam",  "arity": 1, "description": "x passes the exam"}
  ],
  "step2_premises_ast": [
    {
      "premise_id": 0,
      "source_nl": "All students who study hard pass the exam.",
      "ast": {
        "type": "quantifier", "operator": "forall",
        "bound_variables": ["x"],
        "body": {
          "type": "connective", "operator": "implies",
          "operands": [
            {"type": "connective", "operator": "and", "operands": [
              {"type": "predicate", "name": "Student",   "arguments": ["x"]},
              {"type": "predicate", "name": "StudyHard", "arguments": ["x"]}
            ]},
            {"type": "predicate", "name": "PassExam", "arguments": ["x"]}
          ]
        }
      }
    },
    { "premise_id": 1, "source_nl": "Alice is a student.",
      "ast": {"type": "predicate", "name": "Student", "arguments": ["Alice"]} },
    { "premise_id": 2, "source_nl": "Alice studies hard.",
      "ast": {"type": "predicate", "name": "StudyHard", "arguments": ["Alice"]} }
  ]
}
"""

FEW_SHOT_QUESTIONS = """
### VÍ DỤ HOÀN CHỈNH:
Question 1: "Is it true that Alice passes the exam?" (Yes/No)
Question 2: "Which is correct? A. Alice fails. B. Alice passes." (Multiple Choice)

Output:
{
  "step3_question_fol": [
    {
      "question_id": 0,
      "question_type": "yes_no",
      "statement_ast": {"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}
    },
    {
      "question_id": 1,
      "question_type": "multiple_choice",
      "options_ast": {
         "A": {"type": "connective", "operator": "not", "operands": [{"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}]},
         "B": {"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}
      }
    }
  ]
}
"""

# ------------------------------------------------------------------
# PASS 1 PROMPTS (PREMISES ONLY)
# ------------------------------------------------------------------

PREMISE_FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Buoc 1: Tao LOCAL ONTOLOGY -- danh sach Predicates\n"
    "  Buoc 2: Chuyen TUNG tien de thanh cay AST JSON de quy\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    + FEW_SHOT_PREMISES + "\n"
    "Output JSON THUAN TUY (khong markdown, khong text thua):\n"
    '{\n'
    '  "step1_local_ontology": [\n'
    '    {"predicate": "Name", "arity": N, "description": "..."}\n'
    '  ],\n'
    '  "step2_premises_ast": [\n'
    '    {"premise_id": 0, "source_nl": "...", "ast": { <AST> }}\n'
    '  ]\n'
    '}\n'
)

CORRECTION_SYSTEM = (
    "Ban la chuyen gia sua loi FOL. He thong Z3 da phat hien loi.\n"
    "Nhiem vu: sua lai Buoc 1 + Buoc 2 de khong con loi.\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    "Output JSON thuan tuy -- format GIONG HET lan dau.\n"
)

# ------------------------------------------------------------------
# PASS 2 PROMPTS (QUESTIONS ONLY)
# ------------------------------------------------------------------

QUESTION_FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Chuyen TUNG cau hoi thanh AST JSON de kiem tra Z3 Entailment.\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    + FEW_SHOT_QUESTIONS + "\n"
    "QUAN TRONG:\n"
    "  - Ban PHAI su dung cac Predicate tu LOCAL ONTOLOGY duoc cung cap.\n"
    "  - question_type: 'yes_no' hoac 'multiple_choice'\n"
    "  - yes_no: chuyen statement thanh 1 AST node (statement_ast)\n"
    "  - multiple_choice: chuyen TUNG option A/B/C/D thanh AST (options_ast)\n\n"
    "Output JSON THUAN TUY:\n"
    '{\n'
    '  "step3_question_fol": [\n'
    '    {\n'
    '      "question_id": 0,\n'
    '      "question_type": "yes_no",\n'
    '      "source_nl": "statement to check",\n'
    '      "statement_ast": { <AST> }\n'
    '    }\n'
    '  ]\n'
    '}\n'
)

# Fallback
# Fallback -- exact.ipynb CoT format
ANSWER_SYSTEM = (
    "You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).\n"
    "Given a set of premises (in natural language and FOL notation), you must:\n"
    "1. Analyze the logical structure of each premise carefully.\n"
    "2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation.\n"
    "3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.\n"
    "4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.\n"
    "5. For Yes/No questions: verify the statement logically.\n"
    "6. If the premises are INSUFFICIENT, your Final Answer MUST be exactly: Unknown\n"
    "7. Never hallucinate a conclusion not entailed by the premises.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your reasoning here>\n"
    "### Final Answer\n"
    "<A or B or C or D or Yes or No or Unknown>"
)

# Physics
PHYSICS_SOLVER_SYSTEM = (
    "You are an expert physics problem solver.\n"
    "Solve the problem step-by-step, showing all calculations.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your detailed solution here>\n"
    "### Final Answer\n"
    "<your numerical answer with unit>"
)
PHYSICS_MC_SYSTEM = (
    "You are an expert physics problem solver.\n"
    "Solve the multiple-choice problem step-by-step.\n"
    "Evaluate each option against the physics principles.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your detailed solution here>\n"
    "### Final Answer\n"
    "<A or B or C or D or your numerical answer>"
)

# ==================================================================
# UTILS
# ==================================================================
import os, csv, re, time
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

def load_dataset(path, is_physics=False, max_samples=N_SAMPLES, split_mode=RUN_ON_SPLIT):
    if not path or not os.path.exists(path): return []
    if path.endswith(".csv"):
        with open(path, encoding="utf-8") as f: raw = list(csv.DictReader(f))
    else:
        with open(path, encoding="utf-8") as f: raw = json.load(f)
    
    out = raw[:max_samples]
    
    # --- SPLIT LOGIC ---
    import random
    rng = random.Random(SEED)
    rng.shuffle(out)
    n = len(out)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    
    if split_mode == "train":
        out = out[:n_train]
    elif split_mode == "val":
        out = out[n_train:n_train+n_val]
    elif split_mode == "test":
        out = out[n_train+n_val:]
    # if split_mode == "all", keep out as is

    if is_physics and out:
        for s in out:
            s["premises-NL"] = []
            s["questions"]   = [s.get("question", "")]
            s["answers"]     = [str(s.get("answer", "Unknown"))]
            s["_unit"]       = s.get("unit", "")
            s["_is_physics"] = True
    return out

def safe_json(text):
    text = text.strip()
    # v7 FIX: Strip Qwen3 <think>...</think> tags
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    try: return json.loads(text)
    except: pass
    m = re.search(r"```(?:json)?\s*\n?(.*?)```", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1).strip())
        except: pass
    start = text.find("{")
    if start >= 0:
        depth, end = 0, start
        for i in range(start, len(text)):
            if text[i] == "{": depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break
        try: return json.loads(text[start : end + 1])
        except: pass
    return {}

def batch_generate(prompt_pairs, max_tokens, enable_thinking=True, use_lora=False):
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg}, {"role": "user", "content": usr_msg}]
        try:
            formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking))
        except TypeError:
            formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    params = SamplingParams(temperature=0.05, max_tokens=max_tokens, repetition_penalty=1.1)
    _lr = LORA_REQ if (use_lora and LORA_REQ is not None) else None
    outputs = llm.generate(formatted, params, lora_request=_lr) if _lr else llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [o.outputs[0].text.strip() for o in outputs_sorted]

def batch_generate_bon(prompt_pairs, max_tokens, n=BEST_OF_N, enable_thinking=True, use_lora=False):
    """Generate N candidates per prompt using higher temperature."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            formatted.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking))
        except TypeError:
            formatted.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True))

    params = SamplingParams(
        temperature=BON_TEMPERATURE,
        max_tokens=max_tokens,
        repetition_penalty=1.1,
        n=n,
    )
    _lr = LORA_REQ if (use_lora and LORA_REQ is not None) else None
    outputs = llm.generate(formatted, params, lora_request=_lr) if _lr else llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))

    return [
        [o.text.strip() for o in out.outputs]
        for out in outputs_sorted
    ]


def z3_select_best(candidates):
    """
    v8.1: Returns (data, status, func_cache, const_map).
    """
    PRIORITY = {"sat": 4, "unsat": 3, "unknown": 2, "compile_error": 1, "no_ast": 0}
    best_data, best_status, best_score = {}, "no_ast", -1
    best_func_cache = {}
    best_const_map = {}

    for raw in candidates:
        data = safe_json(raw)
        premises_ast = data.get("step2_premises_ast", [])

        if not premises_ast:
            score = PRIORITY["no_ast"]
            status = "no_ast"
            candidate_cache = {}
            candidate_const = {}
        else:
            candidate_cache = {}
            candidate_const = {}
            z3_info = verify_with_z3(premises_ast, func_cache=candidate_cache, const_map=candidate_const)
            status = z3_info["status"]
            score = PRIORITY.get(status, 0)

        if score > best_score:
            best_score = score
            best_status = status
            best_data = data
            best_func_cache = candidate_cache
            best_const_map = candidate_const

        if best_score >= 4:  # sat -> stop early
            break

    return best_data, best_status, best_func_cache, best_const_map




import re

def extract_final_answer(model_output):
    """
    v8.1: Robust answer extraction from exact.ipynb
    Multi-pattern fallback to minimize UNPARSEABLE results.
    Priority: Final Answer block > inline patterns > standalone letters > keyword scan
    """
    text = model_output.strip()

    # Pattern 1: "### Final Answer" block
    match = re.search(
        r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)",
        text, re.IGNORECASE
    )
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        if re.match(r"^unknown", ans, re.IGNORECASE):
            return "Unknown"
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        if re.match(r"^yes", ans, re.IGNORECASE):
            return "Yes"
        if re.match(r"^no\b", ans, re.IGNORECASE):
            return "No"

    # Pattern 2: "answer is X" or "Answer: X"
    match2 = re.search(
        r"(?:answer\s*(?:is|:)\s*)([A-D]|unknown|yes|no)",
        text, re.IGNORECASE
    )
    if match2:
        g = match2.group(1).strip()
        if g.lower() == "unknown": return "Unknown"
        if g.lower() == "yes":    return "Yes"
        if g.lower() == "no":     return "No"
        return g.upper()

    # Pattern 3: Standalone letter near end of text
    last_500 = text[-500:]
    match3 = re.findall(r"\b([A-D])\b", last_500)
    if match3:
        return match3[-1].upper()

    # Pattern 4: Unknown/Yes/No anywhere
    if re.search(r"\bunknown\b", text, re.IGNORECASE):
        return "Unknown"
    if re.search(r"\byes\b", text, re.IGNORECASE):
        return "Yes"
    if re.search(r"\bno\b", text, re.IGNORECASE):
        return "No"

    return "UNPARSEABLE"

print("extract_final_answer v8.1 (from exact.ipynb) san sang")
def extract_physics_answer(model_output):
    """Extract answer from physics CoT response (v8.2)."""
    text = model_output.strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Pattern 1: ### Final Answer block
    match = re.search(r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)", text, re.IGNORECASE)
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        ans = re.sub(r"^(?:the\s+answer\s+is|answer\s*[:=])\s*", "", ans, flags=re.IGNORECASE).strip()
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        return ans
    # Pattern 2: JSON fallback
    data = safe_json(text)
    if data.get("answer"):
        return str(data["answer"]).strip()
    # Pattern 3: "answer is X"
    match2 = re.search(r"answer\s*(?:is|:|=)\s*(.+?)(?:\n|$)", text, re.IGNORECASE)
    if match2:
        return match2.group(1).strip()
    return "Unknown"

print("extract_physics_answer v8.2 san sang")



In [ ]:

# ==================================================================
# STAGE 3 -- Z3 Compiler + Entailment Checker (v8.1 HARDENED)
# ==================================================================
# v8.1 over v7:
#   1. const_map: Deterministic constant->IntVal mapping (no hash collision)
#   2. SAFE fuzzy: case-insensitive + prefix-strip ONLY (no substring!)
#   3. MC elimination: if 3/4 options contradicted, pick the survivor
# v8.1 over v8:
#   - REMOVED dangerous substring matching
#   - Self-Correction will NOT override Qwen fallback with Unknown
# ==================================================================


def get_z3_func(name, arity, func_cache):
    """Get or create Z3 Function with SAFE fuzzy matching.
    v8.1: Only case-insensitive + prefix-strip. NO substring matching."""
    key = f"{name}_{arity}"
    if key in func_cache:
        return func_cache[key]

    # --- v8.1: SAFE fuzzy match (no substring!) ---
    def _normalize(n):
        """Strip Is/Has/Can prefix for matching."""
        n_low = n.lower()
        # v10 FIX: removed can/can_ -- modal verbs (ability/permission) are NOT
        # the same predicate as their bare form (actuality). Caused wrong entailments.
        for prefix in ("is_", "has_", "is", "has"):
            if n_low.startswith(prefix) and len(n_low) > len(prefix):
                return n_low[len(prefix):]
        return n_low

    for cached_key in list(func_cache.keys()):
        parts = cached_key.rsplit("_", 1)
        if len(parts) != 2:
            continue
        cached_name, cached_arity_str = parts
        try:
            cached_arity = int(cached_arity_str)
        except ValueError:
            continue
        if cached_arity != arity:
            continue

        # Level 1: Case-insensitive exact match
        if cached_name.lower() == name.lower():
            func_cache[key] = func_cache[cached_key]
            print(f"    [FUZZY] {name}/{arity} -> {cached_name} (case)")
            return func_cache[key]

        # Level 2: Prefix-strip match (IsStudent -> student == student)
        if _normalize(name) == _normalize(cached_name):
            func_cache[key] = func_cache[cached_key]
            print(f"    [FUZZY] {name}/{arity} -> {cached_name} (prefix-strip)")
            return func_cache[key]

        # v8.1: NO Level 3 substring matching -- it caused false positives!

    # No fuzzy match -- create new function
    sorts = [IntSort()] * arity + [BoolSort()]
    func_cache[key] = Function(name, *sorts)
    return func_cache[key]


def _resolve_bound_var_name(bv):
    if isinstance(bv, dict):
        return bv.get("name", str(bv))
    return str(bv)


def _get_constant_val(name, const_map):
    """v8: Deterministic constant mapping. No hash collisions."""
    if name not in const_map:
        const_map[name] = IntVal(len(const_map) + 1)
    return const_map[name]


def _resolve_predicate_arg(a, var_map, func_cache, const_map):
    if isinstance(a, str):
        if a in var_map:
            return var_map[a]
        return _get_constant_val(a, const_map)
    if isinstance(a, dict):
        atype = a.get("type", "")
        name = a.get("name", "")
        if atype == "variable":
            if name in var_map:
                return var_map[name]
            v = Int(name)
            var_map[name] = v
            return v
        if atype == "constant":
            return _get_constant_val(name, const_map)
        raise ValueError(f"Argument khong hop le (type={atype!r})")
    return _get_constant_val(str(a), const_map)


def compile_ast(node, var_map, func_cache, const_map):
    """Compile 1 AST node -> Z3 expression.
    v8.1: Uses shared func_cache + const_map."""
    if not isinstance(node, dict):
        raise ValueError(f"Expected dict, got {type(node)}: {node!r}")

    ntype = node.get("type", "")

    if ntype == "quantifier":
        op = node.get("operator", "").lower()
        bvs = node.get("bound_variables", [])
        if not bvs:
            raise ValueError("quantifier thieu bound_variables")
        bv_names = [_resolve_bound_var_name(bv) for bv in bvs]
        z3_bvs = [Int(v) for v in bv_names]
        child_map = {**var_map, **{v: z3_bvs[i] for i, v in enumerate(bv_names)}}
        body = compile_ast(node["body"], child_map, func_cache, const_map)
        if op == "forall":
            return ForAll(z3_bvs, body)
        elif op in ("exists", "exist"):
            return Exists(z3_bvs, body)
        else:
            raise ValueError(f"Quantifier khong hop le: {op!r}")

    elif ntype == "connective":
        op = node.get("operator", "").lower()
        ops = [compile_ast(o, var_map, func_cache, const_map) for o in node.get("operands", [])]
        if op == "and":
            return And(*ops)
        elif op == "or":
            return Or(*ops)
        elif op == "implies":
            if len(ops) != 2:
                raise ValueError(f"implies can 2 operands, nhan {len(ops)}")
            return Implies(ops[0], ops[1])
        elif op == "iff":
            if len(ops) != 2:
                raise ValueError(f"iff can 2 operands, nhan {len(ops)}")
            return And(Implies(ops[0], ops[1]), Implies(ops[1], ops[0]))
        elif op == "not":
            if len(ops) != 1:
                raise ValueError(f"not can 1 operand, nhan {len(ops)}")
            return Not(ops[0])
        else:
            raise ValueError(f"Connective khong hop le: {op!r}")

    elif ntype == "predicate":
        name = node.get("name", "")
        args = node.get("arguments", [])
        if not name:
            raise ValueError('predicate thieu "name"')
        func = get_z3_func(name, len(args), func_cache)
        z3_args = [_resolve_predicate_arg(a, var_map, func_cache, const_map) for a in args]
        return func(*z3_args)

    elif ntype in ("variable", "constant"):
        name = node.get("name", "")
        if name in var_map:
            return var_map[name]
        if ntype == "constant":
            return _get_constant_val(name, const_map)
        v = Int(name)
        var_map[name] = v
        return v

    else:
        raise ValueError(f"AST node type khong hop le: {ntype!r}")


def verify_with_z3(premises_ast, func_cache=None, const_map=None):
    """Compile all premises -> Z3, check consistency."""
    if func_cache is None:
        func_cache = {}
    if const_map is None:
        const_map = {}
    solver = Solver()
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast = item.get("ast", {})
            if not ast:
                errors.append(f"Premise {pid}: AST rong")
                continue
            expr = compile_ast(ast, {}, func_cache, const_map)
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"Premise {pid}: {str(e)[:250]}")

    if errors:
        return {"status": "compile_error", "errors": errors,
                "compiled_count": compiled, "total_count": len(premises_ast)}
    try:
        result = solver.check()
        return {"status": str(result), "errors": [],
                "compiled_count": compiled, "total_count": len(premises_ast)}
    except Exception as e:
        return {"status": "solver_error", "errors": [str(e)],
                "compiled_count": compiled, "total_count": len(premises_ast)}


def hallucination_check(local_ontology, premises_ast):
    """Check: all predicates in AST must be in Local Ontology."""
    declared = set()
    for item in local_ontology:
        if isinstance(item, dict):
            declared.add(item.get("predicate", ""))

    warnings = []

    def collect_predicates(node, found):
        if not isinstance(node, dict):
            return
        if node.get("type") == "predicate":
            found.add(node.get("name", ""))
        for v in node.values():
            if isinstance(v, dict):
                collect_predicates(v, found)
            elif isinstance(v, list):
                for sub in v:
                    collect_predicates(sub, found)

    for item in premises_ast:
        used = set()
        collect_predicates(item.get("ast", {}), used)
        hallucinated = used - declared - {""}
        if hallucinated:
            warnings.append(
                f"Premise {item.get('premise_id', '?')}: "
                f"predicates not in Ontology -> {hallucinated}"
            )
    return warnings


# ==================================================================
# Z3 ENTAILMENT CHECKER (v8.1 HARDENED)
# ==================================================================

def _compile_premises_to_solver_shared(premises_ast, func_cache, const_map):
    """Compile premises using SHARED func_cache + const_map."""
    solver = Solver()
    solver.set("timeout", Z3_SOLVER_TIMEOUT)
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast_node = item.get("ast", {})
            if not ast_node:
                continue
            expr = compile_ast(ast_node, {}, func_cache, const_map)
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"P{pid}: {str(e)[:200]}")

    return solver, compiled, errors


def z3_entailment_check(premises_ast, question_fol_item, func_cache=None, const_map=None):
    """v8.1: Shared func_cache + const_map + safe fuzzy matching."""
    if func_cache is None:
        func_cache = {}
    if const_map is None:
        const_map = {}

    q_type = question_fol_item.get("question_type", "yes_no")

    if q_type == "yes_no":
        return _entail_yes_no(premises_ast, question_fol_item, func_cache, const_map)
    elif q_type == "multiple_choice":
        return _entail_mc(premises_ast, question_fol_item, func_cache, const_map)
    else:
        return {"answer": "Unknown", "method": "unsupported_qtype"}


def _entail_yes_no(premises_ast, q_item, func_cache, const_map):
    """Entailment for Yes/No questions."""
    stmt_ast = q_item.get("statement_ast", {})
    if not stmt_ast:
        return {"answer": "Unknown", "method": "no_statement_ast"}

    try:
        stmt_expr = compile_ast(stmt_ast, {}, func_cache, const_map)
    except Exception as e:
        return {"answer": "Unknown", "method": "stmt_compile_error",
                "detail": str(e)[:200]}

    # Test YES: premises + NOT(stmt) -> UNSAT?
    solver1, c1, e1 = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
    if e1:
        return {"answer": "Unknown", "method": "premise_compile_error",
                "detail": "; ".join(e1[:3])}

    try:
        solver1.push()
        solver1.add(Not(stmt_expr))
        r1 = solver1.check()
        solver1.pop()
    except Exception as e:
        r1 = None

    if r1 == unsat:
        return {"answer": "Yes", "method": "z3_entailment",
                "detail": "premises + NOT(Q) is UNSAT => Q is entailed"}

    # Test NO: premises + stmt -> UNSAT?
    solver2, _, _ = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
    try:
        solver2.push()
        solver2.add(stmt_expr)
        r2 = solver2.check()
        solver2.pop()
    except Exception as e:
        r2 = None

    if r2 == unsat:
        return {"answer": "No", "method": "z3_negation",
                "detail": "premises + Q is UNSAT => Q contradicts premises"}

    return {"answer": "Unknown", "method": "z3_undetermined",
            "detail": f"Neither entailed nor contradicted (r1={r1}, r2={r2})"}


def _entail_mc(premises_ast, q_item, func_cache, const_map):
    """Entailment for Multiple Choice (A/B/C/D) with elimination logic."""
    options_ast = q_item.get("options_ast", {})
    if not options_ast:
        return {"answer": "Unknown", "method": "no_options_ast"}

    entailed = []
    consistent = []
    contradicted = []
    details = {}

    for label in ("A", "B", "C", "D"):
        opt_ast = options_ast.get(label, {})
        if not opt_ast:
            continue

        try:
            opt_expr = compile_ast(opt_ast, {}, func_cache, const_map)
        except Exception as e:
            details[label] = f"compile_error: {str(e)[:100]}"
            continue

        solver, c, e = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
        if e:
            details[label] = "premise_error"
            continue

        try:
            # Entailment: premises + NOT(option) -> UNSAT?
            solver.push()
            solver.add(Not(opt_expr))
            r = solver.check()
            solver.pop()

            if r == unsat:
                entailed.append(label)
                details[label] = "entailed"

            # Contradiction: premises + option -> UNSAT?
            solver.push()
            solver.add(opt_expr)
            r2 = solver.check()
            solver.pop()

            if r2 == unsat:
                contradicted.append(label)
                if label not in details:
                    details[label] = "contradicted"
            elif r2 == sat:
                consistent.append(label)
                if label not in details:
                    details[label] = "consistent"
        except Exception as e:
            details[label] = f"solver_error: {str(e)[:100]}"

    # Decision logic (v8 enhanced)
    if len(entailed) == 1:
        return {"answer": entailed[0], "method": "z3_unique_entailment",
                "detail": f"Only {entailed[0]} entailed. {details}"}
    elif len(entailed) > 1:
        return {"answer": entailed[0], "method": "z3_multi_entailment",
                "detail": f"Multiple entailed: {entailed}. {details}"}

    # v8: If only 1 option is NOT contradicted, pick it
    non_contradicted = [l for l in ("A", "B", "C", "D")
                        if l in consistent and l not in contradicted]
    if len(non_contradicted) == 1:
        return {"answer": non_contradicted[0], "method": "z3_elimination",
                "detail": f"Elimination: only {non_contradicted[0]} survives. {details}"}

    if len(consistent) == 1:
        return {"answer": consistent[0], "method": "z3_unique_consistent",
                "detail": f"Only {consistent[0]} consistent. {details}"}
    else:
        return {"answer": "Unknown", "method": "z3_mc_undetermined",
                "detail": f"Entailed={entailed}, Consistent={consistent}, "
                          f"Contradicted={contradicted}. {details}"}


print("Z3 compiler v8.1 (const_map + SAFE fuzzy + elimination) san sang")


In [ ]:
# ==================================================================
# STAGE 2.5 -- Deterministic FOL-string -> Z3 Parser (v11)
# 99.6% formula / 98.8% sample coverage on the challenge dataset.
# ==================================================================
import re
from z3 import (Int, IntSort, BoolSort, Function, ForAll, Exists, IntVal,
                And, Or, Not, Implies, Solver, sat, unsat)

TOKEN_RE = re.compile(r"∀|∃|→|↔|¬|∧|∨|≥|≤|≠|>=|<=|!=|=|>|<|\+|\-|\*|/|\(|\)|,|'[^']*'|\d+\.?\d*|[A-Za-z_][A-Za-z0-9_]*|\S")
CMP = {'=','>','<','≥','≤','≠','>=','<=','!='}
def tokenize(s):
    return TOKEN_RE.findall(s)

class FOLParser:
    """v11 parser: FOL subset + arithmetic comparisons -> Z3.
    Bool functions (predicates) and Int functions (numeric terms) kept in
    separate caches keyed by name_arity."""
    def __init__(self, func_cache, const_map, int_cache=None):
        self.func_cache = func_cache      # Bool-valued predicates
        self.const_map = const_map
        self.int_cache = int_cache if int_cache is not None else {}  # Int-valued funcs

    def parse(self, s):
        self.toks = tokenize(s); self.pos = 0
        self.scope = {}; self.free = {}
        expr = self._iff()
        if self.pos != len(self.toks):
            raise ValueError(f"trailing tokens: {self.toks[self.pos:]}")
        if self.free:
            expr = ForAll(list(self.free.values()), expr)
        return expr

    def _peek(self): return self.toks[self.pos] if self.pos < len(self.toks) else None
    def _eat(self, t=None):
        cur = self._peek()
        if cur is None: raise ValueError("unexpected end")
        if t is not None and cur != t: raise ValueError(f"expected {t}, got {cur}")
        self.pos += 1; return cur

    def _iff(self):
        left = self._implies()
        while self._peek() == '↔':
            self._eat(); right = self._implies()
            left = And(Implies(left, right), Implies(right, left))
        return left
    def _implies(self):
        left = self._or()
        if self._peek() == '→':
            self._eat(); return Implies(left, self._implies())
        return left
    def _or(self):
        left = self._and()
        while self._peek() == '∨':
            self._eat(); left = Or(left, self._and())
        return left
    def _and(self):
        left = self._not()
        while self._peek() == '∧':
            self._eat(); left = And(left, self._not())
        return left
    def _not(self):
        if self._peek() == '¬':
            self._eat(); return Not(self._not())
        return self._quant_or_atom()
    def _quant_or_atom(self):
        t = self._peek()
        if t in ('∀','∃'):
            self._eat(); var = self._eat()
            v = Int(var); saved = self.scope.get(var); self.scope[var] = v
            body = self._not()
            if saved is None: del self.scope[var]
            else: self.scope[var] = saved
            return ForAll([v], body) if t=='∀' else Exists([v], body)
        if t == '(':
            self._eat('('); e = self._iff(); self._eat(')'); return e
        return self._atom()
    def _atom(self):
        # Could be a Bool predicate, or an arithmetic comparison.
        # Try to parse an arithmetic term first; if followed by CMP -> comparison.
        start = self.pos
        term = self._arith()
        if self._peek() in CMP:
            op = self._eat(); rhs = self._arith()
            return self._cmp(op, term, rhs)
        # not a comparison: term must itself be a Bool predicate
        if term is None:
            raise ValueError("empty atom")
        return term  # _arith returns Bool pred when it was a bare predicate

    def _cmp(self, op, a, b):
        if op in ('=',): return a == b
        if op in ('≠','!='): return a != b
        if op in ('>',): return a > b
        if op in ('<',): return a < b
        if op in ('≥','>='): return a >= b
        if op in ('≤','<='): return a <= b
        raise ValueError(f"bad cmp {op}")

    def _arith(self):
        left = self._arith_term()
        while self._peek() in ('+','-'):
            op=self._eat(); right=self._arith_term()
            left = (left+right) if op=='+' else (left-right)
        return left
    def _arith_term(self):
        left = self._factor()
        while self._peek() in ('*','/'):
            op=self._eat(); right=self._factor()
            left = (left*right) if op=='*' else (left/right)
        return left
    def _factor(self):
        tok = self._peek()
        if tok == '(':
            self._eat('('); e=self._iff(); self._eat(')'); return e
        if re.match(r'^\d', tok):
            self._eat()
            return IntVal(int(float(tok)))
        if tok.startswith("'") and tok.endswith("'"):
            self._eat()
            cname = tok.strip("'")
            if cname not in self.const_map:
                self.const_map[cname] = IntVal(len(self.const_map)+1)
            return self.const_map[cname]
        # name: predicate (Bool) or numeric function (Int) depending on following CMP/arith
        name = self._eat()
        args = []
        is_call = False
        if self._peek() == '(':
            is_call=True; self._eat('(')
            if self._peek() != ')':
                args.append(self._arith())
                while self._peek()==',':
                    self._eat(','); args.append(self._arith())
            self._eat(')')
        # Decide Bool vs Int by lookahead: if next is CMP/arith-op, this name is Int-valued
        nxt = self._peek()
        numeric_ctx = nxt in CMP or nxt in ('+','-','*','/')
        if name in self.scope: return self.scope[name]
        if not is_call and re.match(r'^[a-z][a-z0-9_]*$', name) and not args:
            # bare lowercase -> free var (numeric or term)
            if name not in self.free: self.free[name]=Int(name)
            return self.free[name]
        key=f"{name}_{len(args)}"
        if numeric_ctx:
            if key not in self.int_cache:
                sorts=[IntSort()]*len(args)+[IntSort()]
                self.int_cache[key]=Function(name+"_I", *sorts)
            return self.int_cache[key](*args) if args else self.int_cache[key]()
        else:
            if key not in self.func_cache:
                sorts=[IntSort()]*len(args)+[BoolSort()]
                self.func_cache[key]=Function(name, *sorts)
            return self.func_cache[key](*args) if args else self.func_cache[key]()

print("FOL parser v11 ready (handles forall/exists/->/<->/not/and/or + arithmetic + string-literals)")


In [ ]:
# ==================================================================
# STAGE 4 -- v12 Pipeline
# FOL-string Pass-2 + BoN Self-Consistency + Predicate Grounding
# + Confidence Gating (3-tier) + idx Filter + Unsat-Core Explanation
# ==================================================================
import re, time, json
from dataclasses import dataclass, field
from collections import Counter
from z3 import Solver, Not, Bool, sat, unsat
from vllm import SamplingParams

# v13.4: Qwen-CoT generation with LoRA v14 (separate from formalizer LoRA path)
def batch_generate_cot_bon(prompt_pairs, max_tokens, n=5, temperature=0.5, enable_thinking=False):
    """v13.6: Qwen-CoT Best-of-N with LoRA v14. Returns list[list[str]] (N cands/prompt)."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=enable_thinking)
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        formatted.append(text)
    params = SamplingParams(temperature=temperature, top_p=0.95, max_tokens=max_tokens,
                            n=n, repetition_penalty=1.1)
    if COT_LORA_REQ is not None:
        outputs = llm.generate(formatted, params, lora_request=COT_LORA_REQ)
    else:
        outputs = llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [[o.text.strip() for o in out.outputs] for out in outputs_sorted]


def cited_premises(text):
    """Parse premise numbers Qwen cites ('premise 3', 'P3') -> 0-based index set."""
    nums = set()
    for m in re.finditer(r'[Pp]remise\s+(?:number\s+)?(\d+)', text or ''):
        nums.add(int(m.group(1)) - 1)
    for m in re.finditer(r'\bP(\d+)\b', text or ''):
        nums.add(int(m.group(1)) - 1)
    return nums


def compute_z3_core(g, sample, q_idx, rep_fol):
    """Unsat-core premise indices for a YN statement (for semantic-agreement feature)."""
    if not rep_fol:
        return set()
    try:
        stmt = FOLParser(g['fc'], g['cm'], g['ic']).parse(rep_fol)
    except Exception:
        return set()
    idx_list = sample.get('idx', [])
    if USE_IDX_FILTER and q_idx < len(idx_list) and isinstance(idx_list[q_idx], list) and idx_list[q_idx]:
        pwi = [(g['per_idx'][j-1], j-1) for j in idx_list[q_idx] if (j-1) in g['per_idx']]
        if not pwi:
            pwi = [(g['per_idx'][k], k) for k in g['per_idx']]
    else:
        pwi = [(g['per_idx'][k], k) for k in g['per_idx']]
    try:
        return set(unsat_core_premises(pwi, stmt))
    except Exception:
        return set()


def batch_generate_cot(prompt_pairs, max_tokens, enable_thinking=False):
    """Generate one Qwen-CoT response per prompt using v14 LoRA when available."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=enable_thinking)
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        formatted.append(text)
    params = SamplingParams(temperature=0.1, top_p=0.95, max_tokens=max_tokens,
                            repetition_penalty=1.1)
    if COT_LORA_REQ is not None:
        outputs = llm.generate(formatted, params, lora_request=COT_LORA_REQ)
    else:
        outputs = llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [out.outputs[0].text.strip() for out in outputs_sorted]



@dataclass
class PipelineResult:
    sample_id: int
    status: str = "pending"
    z3_status: str = "pending"
    z3_compiled: int = 0
    z3_total: int = 0
    z3_attempts: int = 0
    z3_errors: list = field(default_factory=list)
    hallucination_warn: list = field(default_factory=list)
    local_ontology: list = field(default_factory=list)
    premises_ast: list = field(default_factory=list)
    question_fol: list = field(default_factory=list)
    predicted_answers: list = field(default_factory=list)
    ground_truth: list = field(default_factory=list)
    total_questions: int = 0
    correct_count: int = 0
    time_sec: float = 0.0
    error_log: list = field(default_factory=list)
    answer_source: list = field(default_factory=list)


# ---------------- Question type & embedded-FOL detection ----------------
def is_mc_question(q: str) -> bool:
    return bool(re.search(r'\n\s*[ABCD][\.\)]', q))


def extract_embedded_fol(q: str):
    """If a Yes/No question already contains a FOL Statement, return the FOL string."""
    if not re.search(r'[\u2200\u2203\u2192\u00ac\u2227\u2228\u2194]', q):
        return None
    if is_mc_question(q):
        return None
    parts = re.split(r'[Ss]tatement\s*:', q)
    if len(parts) > 1:
        cand = parts[-1].strip().strip("'\"").strip()
        # strip trailing punctuation like '?' or '.' if at end
        cand = re.sub(r'[\?\.]\s*$', '', cand).strip()
        return cand or None
    return None


# ---------------- Predicate grounding (exact, case-insensitive only) ----------------
def extract_predicates_used(fol_str: str):
    # Predicates are written with name DIRECTLY adjacent to '(' (no space).
    # Variables in '∀x (' have a space before '(' and won't match.
    return set(re.findall(r'([A-Za-z_]\w*)\(', fol_str))


def ground_predicates(fol_str: str, allowed: set):
    """Map predicate names to canonical names in `allowed` via exact case-insensitive match.
    Returns (grounded_str, ungrounded_set). Empty ungrounded_set => fully grounded."""
    used = extract_predicates_used(fol_str)
    ungrounded = set()
    out = fol_str
    allowed_lower = {p.lower(): p for p in allowed}
    for p in used:
        if p in allowed:
            continue
        canon = allowed_lower.get(p.lower())
        if canon and canon != p:
            out = re.sub(rf'\b{re.escape(p)}\b', canon, out)
        elif not canon:
            ungrounded.add(p)
    return out, ungrounded


# ---------------- Pass-2 prompts (FOL-string output) ----------------
PASS2_YN_SYSTEM = (
    "You translate Yes/No questions into First-Order Logic. "
    "Use EXACTLY the same syntax and predicate names as the premises shown below. "
    "Allowed symbols ONLY: \u2200 \u2203 \u2192 \u00ac \u2227 \u2228 \u2194. "
    "Reuse predicate names verbatim from the premises. "
    "Output ONLY the formula on a single line. No quotes, no labels, no explanation."
)

PASS2_MC_SYSTEM = (
    "You translate each of the 4 multiple-choice options into First-Order Logic. "
    "Use EXACTLY the same syntax and predicate names as the premises shown below. "
    "Allowed symbols ONLY: \u2200 \u2203 \u2192 \u00ac \u2227 \u2228 \u2194. "
    "Reuse predicate names verbatim. Output EXACTLY 4 lines:\n"
    "A: <formula>\nB: <formula>\nC: <formula>\nD: <formula>\n"
    "No explanation."
)


def _relevant_premise_fols(sample, q_idx):
    fols = sample.get('premises-FOL', [])
    idx_list = sample.get('idx', [])
    if USE_IDX_FILTER and q_idx < len(idx_list) and isinstance(idx_list[q_idx], list) and idx_list[q_idx]:
        rel = [fols[j-1] for j in idx_list[q_idx] if 0 <= j-1 < len(fols)]
        if rel:
            return rel
    return fols


def _ontology_str(preds: set, max_chars=600):
    return ", ".join(sorted(preds))[:max_chars]


def make_pass2_yn_user(sample, q, q_idx, preds):
    rel = _relevant_premise_fols(sample, q_idx)
    shots = "\n".join(f"- {f}" for f in rel)
    onto = _ontology_str(preds)
    return (f"Relevant premises (FOL syntax to mirror):\n{shots}\n\n"
            f"Available predicate names: {onto}\n\n"
            f"Question: {q}\n\nFormula:")


def make_pass2_mc_user(sample, q, q_idx, preds):
    rel = _relevant_premise_fols(sample, q_idx)
    shots = "\n".join(f"- {f}" for f in rel)
    onto = _ontology_str(preds)
    return (f"Relevant premises (FOL syntax to mirror):\n{shots}\n\n"
            f"Available predicate names: {onto}\n\n"
            f"Question (multiple choice):\n{q}\n\n"
            f"Write each option as a FOL formula:")


# ---------------- Parse Pass-2 outputs ----------------
def parse_yn_output(raw: str) -> str:
    """Extract a single FOL formula from a (possibly noisy) Yes/No Pass-2 output."""
    raw = (raw or "").strip()
    for line in reversed(raw.split('\n')):
        line = line.strip().strip('`').strip()
        line = re.sub(r'^(?:formula|fol|answer|statement)\s*[:=]\s*', '', line, flags=re.I)
        if re.search(r'[\u2200\u2203\u2192\u00ac\u2227\u2228\u2194]', line) or re.search(r'[A-Za-z_]\w*\s*\(', line):
            return line
    # last resort: last non-empty line
    lines = [l for l in raw.split('\n') if l.strip()]
    return lines[-1].strip() if lines else ""


def parse_mc_output(raw: str) -> dict:
    """Extract a dict {A,B,C,D} -> FOL string from a Pass-2 MC output."""
    out = {}
    for line in (raw or "").split('\n'):
        m = re.match(r'^\s*([ABCD])\s*[:\.\)]\s*(.+)$', line.strip())
        if m:
            f = m.group(2).strip().strip('`').strip()
            out[m.group(1)] = f
    return out


# ---------------- Premise FOL parsing ----------------
def parse_gold_premises(fol_strings):
    fc, cm, ic = {}, {}, {}
    exprs, per_idx = [], {}
    n_ok = 0
    for i, f in enumerate(fol_strings):
        try:
            e = FOLParser(fc, cm, ic).parse(f)
            exprs.append(e)
            per_idx[i] = e
            n_ok += 1
        except Exception:
            pass
    all_ok = (n_ok == len(fol_strings)) and len(fol_strings) > 0
    preds = set(k.rsplit('_', 1)[0] for k in fc.keys() if not k.startswith('__'))
    return dict(exprs=exprs, per_idx=per_idx, fc=fc, cm=cm, ic=ic,
                preds=preds, all_ok=all_ok, n_ok=n_ok, n_tot=len(fol_strings))


def _select_premises(g, sample, q_idx):
    if not USE_IDX_FILTER:
        return g['exprs']
    idx_list = sample.get('idx', [])
    if q_idx < len(idx_list) and isinstance(idx_list[q_idx], list) and idx_list[q_idx]:
        rel = [g['per_idx'][j-1] for j in idx_list[q_idx] if (j-1) in g['per_idx']]
        if rel:
            return rel
    return g['exprs']


# ---------------- Entailment ----------------
def _solver_with(exprs):
    s = Solver(); s.set('timeout', Z3_SOLVER_TIMEOUT)
    for e in exprs:
        s.add(e)
    return s


def entail_yn(premise_exprs, stmt):
    s = _solver_with(premise_exprs)
    s.push(); s.add(Not(stmt))
    try:
        r1 = s.check()
    except Exception:
        r1 = None
    s.pop()
    if r1 == unsat:
        return 'Yes'
    s.push(); s.add(stmt)
    try:
        r2 = s.check()
    except Exception:
        r2 = None
    s.pop()
    if r2 == unsat:
        return 'No'
    return 'Unknown'


def entail_mc(premise_exprs, options_exprs):
    if not options_exprs:
        return 'Unknown'
    entailed, consistent, contradicted = [], [], []
    for lab, oe in options_exprs.items():
        s = _solver_with(premise_exprs)
        s.push(); s.add(Not(oe))
        try: r1 = s.check()
        except Exception: r1 = None
        s.pop()
        if r1 == unsat:
            entailed.append(lab)
        s.push(); s.add(oe)
        try: r2 = s.check()
        except Exception: r2 = None
        s.pop()
        if r2 == unsat:
            contradicted.append(lab)
        elif r2 == sat:
            consistent.append(lab)
    if len(entailed) == 1:
        return entailed[0]
    if len(entailed) > 1:
        return entailed[0]
    nonc = [l for l in options_exprs if l in consistent and l not in contradicted]
    if len(nonc) == 1:
        return nonc[0]
    if len(consistent) == 1:
        return consistent[0]
    return 'Unknown'


# ---------------- Unsat-core explanation (XAI deliverable) ----------------
def unsat_core_premises(premise_exprs_with_idx, stmt):
    """premise_exprs_with_idx: list of (premise_z3_expr, original_index).
    Returns list of original indices whose premises were used in the proof, or []."""
    s = Solver(); s.set('timeout', Z3_SOLVER_TIMEOUT)
    bools = []
    for k, (e, orig_i) in enumerate(premise_exprs_with_idx):
        b = Bool(f'__core_p{k}')
        s.assert_and_track(e, b)
        bools.append((b, orig_i))
    s.add(Not(stmt))
    try:
        if s.check() == unsat:
            core_strs = set(str(c) for c in s.unsat_core())
            return [orig_i for (b, orig_i) in bools if str(b) in core_strs]
    except Exception:
        pass
    return []


# ---------------- Self-consistency vote ----------------
def sc_vote_yn(premise_exprs, candidates, preds, fc, cm, ic):
    """Vote over Yes/No/Unknown from BoN FOL-string candidates."""
    votes = Counter()
    n_parsed = 0
    grounded_fols = []
    for cand in candidates:
        if not cand:
            votes['_empty'] += 1
            continue
        grounded, ung = ground_predicates(cand, preds)
        if ung:
            votes['_ungrounded'] += 1
            continue
        try:
            stmt = FOLParser(fc, cm, ic).parse(grounded)
        except Exception:
            votes['_parse_err'] += 1
            continue
        n_parsed += 1
        grounded_fols.append((grounded, stmt))
        votes[entail_yn(premise_exprs, stmt)] += 1
    return votes, n_parsed, grounded_fols


def sc_vote_mc(premise_exprs, candidate_sets, preds, fc, cm, ic):
    """Vote over A/B/C/D/Unknown from BoN candidate sets."""
    votes = Counter()
    n_parsed = 0
    for cand in candidate_sets:
        if not isinstance(cand, dict) or len(cand) < 2:
            votes['_empty'] += 1
            continue
        opt_exprs = {}
        ok = True
        for lab in 'ABCD':
            if lab not in cand:
                continue
            g, ung = ground_predicates(cand[lab], preds)
            if ung:
                ok = False; break
            try:
                opt_exprs[lab] = FOLParser(fc, cm, ic).parse(g)
            except Exception:
                ok = False; break
        if not ok or len(opt_exprs) < 2:
            votes['_failed'] += 1
            continue
        n_parsed += 1
        votes[entail_mc(premise_exprs, opt_exprs)] += 1
    return votes, n_parsed


def gate_decision(votes: Counter, total_candidates: int):
    """3-tier gate: returns (best_answer, confidence, tier).
    tier in {'z3_trust', 'z3_arbiter', 'fallback'}."""
    real = {k: v for k, v in votes.items() if not k.startswith('_')}
    if not real:
        return None, 0.0, 'fallback'
    total = sum(real.values())
    ans, c = max(real.items(), key=lambda x: x[1])
    conf = c / total
    grounded_frac = total / max(total_candidates, 1)
    definite = ans not in (None, 'Unknown')
    if definite and conf >= SC_CONFIDENCE_TAU and grounded_frac >= GROUNDED_FRAC_TAU:
        return ans, conf, 'z3_trust'
    if definite and conf >= 0.5:
        return ans, conf, 'z3_arbiter'
    return ans, conf, 'fallback'


# ---------------- BoN generation helper ----------------
def batch_generate_n(prompt_pairs, max_tokens, n=5, temperature=0.7, enable_thinking=False):
    """Generate N samples per prompt using vLLM n parameter. Returns list[list[str]]."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=enable_thinking)
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        formatted.append(text)
    params = SamplingParams(
        temperature=temperature, top_p=0.95, max_tokens=max_tokens, n=n,
        repetition_penalty=1.1,
    )
    outputs = llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [[o.text.strip() for o in out.outputs] for out in outputs_sorted]


# ---------------- Answer-fallback prompt (reuse from utils) ----------------
def _make_answer_user(sample, question):
    lines = ["### Premises"]
    for i, p in enumerate(sample.get('premises-NL', [])):
        lines.append(f"P{i+1}. {p}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"


def _make_informed_answer_user(sample, question, z3_hint, z3_conf, votes_summary, has_z3):
    """v12.1 -- include premise FOL and Z3 evidence so Qwen can reason informedly."""
    nls = sample.get('premises-NL', [])
    fols = sample.get('premises-FOL', [])
    lines = ["### Premises (NL + FOL)"]
    for i, p in enumerate(nls):
        lines.append(f"P{i+1}. {p}")
        if i < len(fols) and fols[i]:
            lines.append(f"      FOL: {fols[i]}")
    out = "\n".join(lines) + f"\n\n### Question\n{question}"
    if has_z3 and z3_hint:
        out += "\n\n### Symbolic evidence (Z3 evaluated 5 formalizations)"
        out += f"\n- Vote distribution: {votes_summary or '(empty)'}"
        out += f"\n- Z3's tentative answer: {z3_hint} (confidence {z3_conf:.2f})"
        out += ("\n- Note: Z3 may over-claim entailment on statements that are "
                "converses/inverses of premises or have unscoped free variables. "
                "Verify carefully by reasoning step-by-step.")
    return out




# ============================== STRUCTURAL FEATURES ==============================
def _split_top_implication(fol_str):
    """Split a FOL string on its top-level '->' (outside any parens). Returns (ante, cons) or None."""
    depth = 0
    for i, ch in enumerate(fol_str):
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
        elif ch == '\u2192' and depth == 0:
            return fol_str[:i], fol_str[i+1:]
    # strip one outer quantifier+paren layer and retry once
    m = re.match(r'^\s*[\u2200\u2203]\w+\s*\((.*)\)\s*$', fol_str)
    if m:
        return _split_top_implication(m.group(1))
    return None


def is_converse_or_inverse(stmt_fol, premise_fols):
    """Heuristic: stmt is A->B; some premise is B->A (converse) or shares
    swapped antecedent/consequent predicate sets (inverse via negation)."""
    s = _split_top_implication(stmt_fol)
    if not s:
        return False
    sa = extract_predicates_used(s[0])
    sb = extract_predicates_used(s[1])
    if not sa or not sb:
        return False
    for pf in premise_fols:
        p = _split_top_implication(pf)
        if not p:
            continue
        pa = extract_predicates_used(p[0])
        pb = extract_predicates_used(p[1])
        if sa == pb and sb == pa and sa != sb:
            return True
    return False


def _has_free_var(fol_str):
    """Variable appears in an atom but no matching quantifier binds it (rough)."""
    quantified = set(re.findall(r'[\u2200\u2203](\w+)', fol_str))
    # vars used as args: lowercase tokens inside parens
    used_vars = set(re.findall(r'[\(,]\s*([a-z]\w*)\s*[\),]', fol_str))
    return bool(used_vars - quantified)


def structural_features(stmt_fol, premise_fols):
    if not stmt_fol:
        return dict(has_free_var=0, is_converse=0, n_quantifiers=0, stmt_len=0)
    return dict(
        has_free_var=int(_has_free_var(stmt_fol)),
        is_converse=int(is_converse_or_inverse(stmt_fol, premise_fols)),
        n_quantifiers=stmt_fol.count('\u2200') + stmt_fol.count('\u2203'),
        stmt_len=len(stmt_fol),
    )


def pick_representative_fol(cands, preds, fc, cm, ic, prem):
    """Return one grounded, parseable FOL string from BoN candidates (for features/explanation)."""
    for c in cands:
        if not c:
            continue
        g, ung = ground_predicates(c, preds)
        if ung:
            continue
        try:
            FOLParser(fc, cm, ic).parse(g)
            return g
        except Exception:
            continue
    return ""


# ============================== FEATURE-EXTRACTION PIPELINE ==============================
def build_features(samples, dataset_name="Dataset"):
    """Run Z3 BoN + entailment AND full Qwen-CoT on EVERY question.
    Returns (records, results_skeleton)."""
    t0 = time.time()
    N = len(samples)
    results = [PipelineResult(sample_id=i,
                              ground_truth=s.get('answers', []),
                              total_questions=len(s.get('questions', [])))
               for i, s in enumerate(samples)]

    # ---- Stage A: premises ----
    print("\n" + "=" * 65 + "\n  STAGE A -- Premise FOL parse\n" + "=" * 65)
    gold = {}
    for i, s in enumerate(samples):
        g = parse_gold_premises(s.get('premises-FOL', []))
        gold[i] = g
        results[i].z3_compiled = g['n_ok']; results[i].z3_total = g['n_tot']
        results[i].z3_status = 'sat' if (g['exprs'] and (g['all_ok'] or g['n_ok'] >= max(1, g['n_tot']-1))) else 'no_ast'
    print("  Premise status:", dict(Counter(r.z3_status for r in results)))

    # ---- Stage B: dispatch ----
    embedded, yn_tasks, mc_tasks = [], [], []
    all_questions = []   # (i, q_idx, q, q_type)
    for i, s in enumerate(samples):
        for q_idx, q in enumerate(s.get('questions', [])):
            qt = 'mc' if is_mc_question(q) else 'yes_no'
            all_questions.append((i, q_idx, q, qt))
            if results[i].z3_status != 'sat':
                continue
            emb = extract_embedded_fol(q)
            if emb:
                embedded.append((i, q_idx, emb))
            elif qt == 'mc':
                mc_tasks.append((i, q_idx, q))
            else:
                yn_tasks.append((i, q_idx, q))
    print(f"  embedded:{len(embedded)} yn:{len(yn_tasks)} mc:{len(mc_tasks)} total_q:{len(all_questions)}")

    # ---- Stage C: Z3 Pass-2 BoN ----
    print("\n" + "=" * 65 + f"\n  STAGE C -- Z3 Pass-2 BoN (N={BON_N_QFORMALIZE})\n" + "=" * 65)
    yn_cands, mc_cands = {}, {}
    if yn_tasks:
        prompts = [(PASS2_YN_SYSTEM, make_pass2_yn_user(samples[i], q, q_idx, gold[i]['preds']))
                   for (i, q_idx, q) in yn_tasks]
        gen = batch_generate_n(prompts, MAX_PASS2_TOKENS_YN, n=BON_N_QFORMALIZE, temperature=BON_TEMP, enable_thinking=False)
        for k, c in enumerate(gen):
            i, q_idx, _ = yn_tasks[k]; yn_cands[(i, q_idx)] = [parse_yn_output(x) for x in c]
    if mc_tasks:
        prompts = [(PASS2_MC_SYSTEM, make_pass2_mc_user(samples[i], q, q_idx, gold[i]['preds']))
                   for (i, q_idx, q) in mc_tasks]
        gen = batch_generate_n(prompts, MAX_PASS2_TOKENS_MC, n=BON_N_QFORMALIZE, temperature=BON_TEMP, enable_thinking=False)
        for k, c in enumerate(gen):
            i, q_idx, _ = mc_tasks[k]; mc_cands[(i, q_idx)] = [parse_mc_output(x) for x in c]
    print(f"  Z3 BoN done. yn={len(yn_cands)} mc={len(mc_cands)}")

    # ---- Stage C2: compute Z3 signals per question ----
    z3_sig = {}   # (i,q_idx) -> dict
    for (i, q_idx, fol) in embedded:
        g = gold[i]; prem = _select_premises(g, samples[i], q_idx)
        gg, ung = ground_predicates(fol, g['preds'])
        if ung:
            z3_sig[(i, q_idx)] = dict(answer='Unknown', conf=0.0, grounded=0.0, definite=False,
                                      spread=0, rep_fol='', source='embed_ungrounded'); continue
        try:
            stmt = FOLParser(g['fc'], g['cm'], g['ic']).parse(gg)
            ans = entail_yn(prem, stmt)
            z3_sig[(i, q_idx)] = dict(answer=ans, conf=1.0, grounded=1.0, definite=(ans != 'Unknown'),
                                      spread=1, rep_fol=gg, source='embed')
        except Exception:
            z3_sig[(i, q_idx)] = dict(answer='Unknown', conf=0.0, grounded=0.0, definite=False,
                                      spread=0, rep_fol='', source='embed_err')
    for (i, q_idx, q) in yn_tasks:
        g = gold[i]; prem = _select_premises(g, samples[i], q_idx)
        cands = yn_cands.get((i, q_idx), [])
        votes, n_parsed, gfols = sc_vote_yn(prem, cands, g['preds'], g['fc'], g['cm'], g['ic'])
        ans, conf, tier = gate_decision(votes, len(cands))
        real = {k: v for k, v in votes.items() if not k.startswith('_')}
        z3_sig[(i, q_idx)] = dict(answer=ans or 'Unknown', conf=conf,
                                  grounded=(n_parsed/max(len(cands), 1)),
                                  definite=(ans not in (None, 'Unknown')),
                                  spread=len(real),
                                  rep_fol=(gfols[0][0] if gfols else pick_representative_fol(cands, g['preds'], g['fc'], g['cm'], g['ic'], prem)),
                                  source='yn')
    for (i, q_idx, q) in mc_tasks:
        g = gold[i]; prem = _select_premises(g, samples[i], q_idx)
        cands = mc_cands.get((i, q_idx), [])
        votes, n_parsed = sc_vote_mc(prem, cands, g['preds'], g['fc'], g['cm'], g['ic'])
        ans, conf, tier = gate_decision(votes, len(cands))
        real = {k: v for k, v in votes.items() if not k.startswith('_')}
        z3_sig[(i, q_idx)] = dict(answer=ans or 'Unknown', conf=conf,
                                  grounded=(n_parsed/max(len(cands), 1)),
                                  definite=(ans not in (None, 'Unknown')),
                                  spread=len(real), rep_fol='', source='mc')

    # ---- Stage D: Qwen-CoT BoN + Self-Consistency (v13.6 Option B) ----
    print("\n" + "=" * 65 + f"\n  STAGE D -- Qwen-CoT BoN (N={QWEN_BON_N}, {len(all_questions)} questions)\n" + "=" * 65)
    cot_prompts = [(V14_COT_SYSTEM, _make_answer_user(samples[i], q)) for (i, q_idx, q, qt) in all_questions]
    cot_gen = batch_generate_cot_bon(cot_prompts, COT_MAX_TOKENS, n=QWEN_BON_N,
                                     temperature=QWEN_BON_TEMP, enable_thinking=False)
    qwen_map = {}   # (i,q_idx) -> dict(answer, conf, spread, text)
    def _norm(a):
        a = str(a).strip()
        return a.upper() if a.lower() in ('yes', 'no', 'unknown') else a.upper()
    for k, cands in enumerate(cot_gen):
        i, q_idx, q, qt = all_questions[k]
        ans_list, text_by_ans = [], {}
        for c in cands:
            a = extract_final_answer(c)
            if a == 'UNPARSEABLE':
                a = safe_json(c).get('answer', 'Unknown')
            a = _norm(a)
            ans_list.append(a)
            text_by_ans.setdefault(a, c)
        vote = Counter(ans_list)
        if vote:
            top_ans, top_n = vote.most_common(1)[0]
            conf = top_n / max(len(ans_list), 1)
            spread = len(vote)
            text = text_by_ans.get(top_ans, cands[0] if cands else "")
        else:
            top_ans, conf, spread, text = 'Unknown', 0.0, 0, ""
        qwen_map[(i, q_idx)] = dict(answer=top_ans, conf=conf, spread=spread, text=text)
    # backward-compat alias
    qwen_ans_map = {k: v['answer'] for k, v in qwen_map.items()}

    # ---- Stage E: assemble feature records ----
    records = []
    for (i, q_idx, q, qt) in all_questions:
        gt = samples[i].get('answers', [])
        gold_ans = gt[q_idx] if q_idx < len(gt) else '?'
        z = z3_sig.get((i, q_idx), dict(answer='Unknown', conf=0.0, grounded=0.0,
                                        definite=False, spread=0, rep_fol='', source='no_premise'))
        qm = qwen_map.get((i, q_idx), dict(answer='Unknown', conf=0.0, spread=0, text=''))
        qa = qm['answer']
        idx_list = samples[i].get('idx', [])
        n_idx = len(idx_list[q_idx]) if (q_idx < len(idx_list) and isinstance(idx_list[q_idx], list)) else 0
        sf = structural_features(z['rep_fol'], samples[i].get('premises-FOL', []))
        agree = int(str(z['answer']).strip().upper() == str(qa).strip().upper())
        # v13.6 Option C: semantic agreement between Z3 unsat-core and Qwen citations
        qwen_cited = cited_premises(qm['text'])
        z3_core = compute_z3_core(gold[i], samples[i], q_idx, z['rep_fol'])
        if z3_core or qwen_cited:
            inter = len(z3_core & qwen_cited); uni = len(z3_core | qwen_cited)
            jacc = inter / uni if uni else 0.0
        else:
            jacc = 0.0
        records.append(dict(
            sample_id=i, q_idx=q_idx, q_type=qt, gold=str(gold_ans),
            z3_answer=str(z['answer']), z3_conf=round(float(z['conf']), 4),
            z3_grounded=round(float(z['grounded']), 4), z3_definite=int(z['definite']),
            z3_spread=int(z['spread']), z3_source=z['source'],
            qwen_answer=str(qa), z3_qwen_agree=agree,
            qwen_sc_conf=round(float(qm['conf']), 4), qwen_vote_spread=int(qm['spread']),
            qwen_n_cited=len(qwen_cited), z3_qwen_jaccard=round(float(jacc), 4),
            q_type_mc=int(qt == 'mc'), n_idx_premises=n_idx,
            **sf,
            z3_correct=int(str(z['answer']).strip().upper() == str(gold_ans).strip().upper()),
            qwen_correct=int(str(qa).strip().upper() == str(gold_ans).strip().upper()),
        ))

    print(f"\n  Built {len(records)} feature records in {time.time()-t0:.1f}s")
    return records, results, gold


print("v13.6 feature-extraction pipeline loaded (Qwen BoN-SC + premise-overlap features)")


In [ ]:

import json, time, random, os
import numpy as np

# Try cache first (any v13.x run produces it)
records = None
if LOAD_FEATURES_CACHE and os.path.exists(FEATURES_CACHE_PATH) and not FORCE_REBUILD:
    try:
        records = json.load(open(FEATURES_CACHE_PATH, encoding="utf-8"))
        print(f"Loaded {len(records)} cached feature records from {FEATURES_CACHE_PATH}")
    except Exception as e:
        print(f"Cache load failed ({e}); will rebuild")
        records = None

if records is None:
    logic_samples = load_dataset(DATASET_PATH, is_physics=False)
    print(f"Building features from scratch on {len(logic_samples)} samples...")
    if len(logic_samples) == 0:
        raise RuntimeError(
            f"Dataset returned 0 samples from DATASET_PATH={DATASET_PATH}\n"
            "Hay add dataset 'logic-based-educational-queries' vao Kaggle session nay.")
    records, _, _ = build_features(logic_samples, "Logic")
    json.dump(records, open(FEATURES_CACHE_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"Saved cache: {FEATURES_CACHE_PATH}")

n_samples = max(r["sample_id"] for r in records) + 1
print(f"Feature records: {len(records)}  |  samples: {n_samples}")

# Deterministic train/val split BY SAMPLE (matches across v13.x)
rng = random.Random(CAL_SEED)
ids = list(range(n_samples)); rng.shuffle(ids)
n_tr = int(n_samples * CAL_TRAIN_RATIO)
train_ids = set(ids[:n_tr]); val_ids = set(ids[n_tr:])
train_recs = [r for r in records if r["sample_id"] in train_ids]
val_recs   = [r for r in records if r["sample_id"] in val_ids]
print(f"Split: train={len(train_recs)} q ({len(train_ids)} samples) | val={len(val_recs)} q ({len(val_ids)} samples)")


def accuracy(recs, decider, **kw):
    if not recs: return (0.0, 0, 0)
    c = sum(1 for r in recs if str(decider(r, **kw)).upper() == str(r["gold"]).upper())
    return (c/len(recs), c, len(recs))


def fmt_row(name, va, fa):
    return (f"  {name:32s} | {va[0]:>6.1%} ({va[1]:>3}/{va[2]:<3}) | "
            f"{fa[0]:>6.1%} ({fa[1]:>3}/{fa[2]:<3})")


# ==================================================================
# v13.3 -- NHANH C: Full comparison of all four methods on held-out val
# pure_qwen | z3_first | parallel_hybrid | calibrator
# ==================================================================
print("\n" + "="*70)
print("  v13.6 -- Qwen BoN-SC (B) + rich calibrator features (C)")
print("="*70)

# ---- Deciders ----
def dec_pure_qwen(r): return r["qwen_answer"]
def dec_pure_z3(r):   return r["z3_answer"] if r["z3_definite"] else r["qwen_answer"]
def dec_z3_first(r):
    if r["z3_definite"] and r["z3_conf"]>=SC_CONFIDENCE_TAU and r["z3_grounded"]>=GROUNDED_FRAC_TAU:
        return r["z3_answer"]
    return r["qwen_answer"]
def dec_parallel_hybrid(r):
    z, q = r["z3_answer"], r["qwen_answer"]
    if not r["z3_definite"]: return q
    if str(z).upper() == str(q).upper(): return z
    if r["is_converse"] or r["has_free_var"]: return q
    if r["z3_conf"] >= PH_HIGH_CONF: return z
    return q
def dec_oracle(r):  # upper bound
    return r["z3_answer"] if r["z3_correct"] else r["qwen_answer"]
def dec_conf_hybrid(r):  # v13.6 Option B+C: confidence-weighted routing
    if not r["z3_definite"]:
        return r["qwen_answer"]
    if str(r["z3_answer"]).upper() == str(r["qwen_answer"]).upper():
        return r["z3_answer"]
    # disagreement: trust Qwen only if its SC-confidence is high AND beats Z3 conf
    if r.get("qwen_sc_conf", 0) >= QWEN_CONF_TRUST and r.get("qwen_sc_conf", 0) > r["z3_conf"]:
        return r["qwen_answer"]
    return r["z3_answer"]

# ---- Train calibrator ----
FEATURES = ["z3_conf","z3_grounded","z3_definite","z3_spread","z3_qwen_agree",
            "q_type_mc","n_idx_premises","has_free_var","is_converse",
            "n_quantifiers","stmt_len",
            "qwen_sc_conf","qwen_vote_spread","qwen_n_cited","z3_qwen_jaccard"]  # v13.6
def to_mat(recs):
    X = np.array([[r[f] for f in FEATURES] for r in recs], dtype=float)
    y = np.array([r["z3_correct"] for r in recs], dtype=int)
    return X, y
X_tr, y_tr = to_mat(train_recs)

backend = None
try:
    import lightgbm as lgb
    model = lgb.LGBMClassifier(n_estimators=300, num_leaves=15, learning_rate=0.04,
                               min_child_samples=10, subsample=0.9, colsample_bytree=0.9,
                               random_state=CAL_SEED, verbose=-1)
    model.fit(X_tr, y_tr); backend = "lightgbm"
except Exception:
    from sklearn.ensemble import GradientBoostingClassifier
    model = GradientBoostingClassifier(n_estimators=300, max_depth=3,
                                       learning_rate=0.04, random_state=CAL_SEED)
    model.fit(X_tr, y_tr); backend = "sklearn-gbm"

p_tr  = model.predict_proba(X_tr)[:, 1]
p_val = model.predict_proba(to_mat(val_recs)[0])[:, 1]
p_full = model.predict_proba(to_mat(records)[0])[:, 1]

best_tau, best_acc = 0.5, 0.0
for tau in [x/100 for x in range(20, 95, 5)]:
    c = sum(1 for j, r in enumerate(train_recs)
            if str(r["z3_answer"] if (r["z3_definite"] and p_tr[j] >= tau) else r["qwen_answer"]).upper()
            == str(r["gold"]).upper())
    a = c / len(train_recs)
    if a > best_acc: best_acc, best_tau = a, tau

def dec_calibrator_val(j, r):  return r["z3_answer"] if (r["z3_definite"] and p_val[j] >= best_tau)  else r["qwen_answer"]
def dec_calibrator_full(j, r): return r["z3_answer"] if (r["z3_definite"] and p_full[j] >= best_tau) else r["qwen_answer"]

cal_val_c  = sum(1 for j, r in enumerate(val_recs)  if str(dec_calibrator_val(j, r)).upper()  == str(r["gold"]).upper())
cal_full_c = sum(1 for j, r in enumerate(records)    if str(dec_calibrator_full(j, r)).upper() == str(r["gold"]).upper())

print(f"  Calibrator backend={backend} | tuned tau={best_tau:.2f} (train acc {best_acc:.1%})")
try:
    imp = model.feature_importances_; order = np.argsort(imp)[::-1]
    print("  Top features:", ", ".join(f"{FEATURES[k]}={imp[k]:.0f}" for k in order[:6]))
except Exception: pass

# ---- Eval all ----
methods = [
    ("pure_qwen_LoRA (v14 CoT)", dec_pure_qwen),
    ("pure_z3 (Z3 when definite)", dec_pure_z3),
    ("z3_first (v12 baseline)", dec_z3_first),
    ("PARALLEL_HYBRID (rule)", dec_parallel_hybrid),
    ("CONF_HYBRID (v13.6 B+C)", dec_conf_hybrid),
]
print("\n" + "="*70)
print(f"  {'METHOD':32s} | {'VAL':>14s} | {'FULL':>14s}")
print("-"*70)
rows = {}
for name, dec in methods:
    va = accuracy(val_recs, dec); fa = accuracy(records, dec)
    rows[name] = (va, fa)
    print(fmt_row(name, va, fa))
print(fmt_row("CALIBRATOR (nhanh A, LGBM)",
              (cal_val_c/len(val_recs), cal_val_c, len(val_recs)),
              (cal_full_c/len(records), cal_full_c, len(records))))
print("-"*70)
orc_v = accuracy(val_recs, dec_oracle); orc_f = accuracy(records, dec_oracle)
print(fmt_row("ORACLE (upper bound)", orc_v, orc_f))
print("="*70)

# ---- Diagnostic ----
dis = [r for r in val_recs if str(r["z3_answer"]).upper()!=str(r["qwen_answer"]).upper() and r["z3_definite"]]
z3_w = sum(1 for r in dis if r["z3_correct"] and not r["qwen_correct"])
qw_w = sum(1 for r in dis if r["qwen_correct"] and not r["z3_correct"])
bw   = sum(1 for r in dis if not r["z3_correct"] and not r["qwen_correct"])
conv = [r for r in dis if r["is_converse"]]
print(f"\n  VAL disagreements (z3 definite): {len(dis)}")
print(f"    Z3 right / Qwen wrong : {z3_w}")
print(f"    Qwen right / Z3 wrong : {qw_w}")
print(f"    both wrong            : {bw}")
print(f"    converse-template     : {len(conv)} (Qwen right {sum(1 for r in conv if r['qwen_correct'])}/{len(conv)})")

# ---- Save ----
summary = {
    "meta":{"version":"v13_6_bon_sc_richfeatures","backend":backend,"tau":best_tau,
            "train_samples":len(train_ids),"val_samples":len(val_ids),
            "timestamp":time.strftime("%Y-%m-%d %H:%M:%S")},
    "val":  {n:dict(acc=r[0][0],correct=r[0][1],total=r[0][2]) for n,r in rows.items()},
    "full": {n:dict(acc=r[1][0],correct=r[1][1],total=r[1][2]) for n,r in rows.items()},
    "calibrator":{"val":dict(acc=cal_val_c/len(val_recs),correct=cal_val_c,total=len(val_recs)),
                  "full":dict(acc=cal_full_c/len(records),correct=cal_full_c,total=len(records))},
    "oracle":{"val":dict(acc=orc_v[0],correct=orc_v[1],total=orc_v[2]),
              "full":dict(acc=orc_f[0],correct=orc_f[1],total=orc_f[2])},
    "disagreement":{"val_total":len(dis),"z3_wins":z3_w,"qwen_wins":qw_w,"both_wrong":bw,
                    "converse":len(conv),"converse_qwen_right":sum(1 for r in conv if r["qwen_correct"])},
}
out_path = "/kaggle/working/pipeline_results_v13_6_compare.json"
json.dump(summary, open(out_path,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"\nSaved: {out_path}")
print("\n" + "="*70 + "\n  v13.6 HOAN TAT\n" + "="*70)


In [ ]:
# ==================================================================
# DIAGNOSTIC CELL — paste vào cuối notebook_v13_5_v4 / v13_6_v4
# Không cần GPU, không cần LLM, chạy từ feature cache
# ==================================================================
import json, os, numpy as np
from collections import Counter

# ---- 0. Load feature cache (tự động tìm) ----
CACHE_CANDIDATES = [
    "/kaggle/working/pipeline_features_v136_v4.json",
    "/kaggle/working/pipeline_features_v135_v4.json",
    "/kaggle/working/pipeline_features_v13_5.json",
    "/kaggle/working/pipeline_features_v13_6.json",
    FEATURES_CACHE_PATH,   # biến đã có trong notebook
]
records = None
for p in CACHE_CANDIDATES:
    try:
        if os.path.exists(p):
            records = json.load(open(p, encoding="utf-8"))
            print(f"Loaded cache: {p}  ({len(records)} records)")
            break
    except Exception: pass
if records is None:
    raise RuntimeError("Không tìm thấy feature cache. Chạy build_features trước.")

def norm(x): return str(x).strip().upper()
yn_map = {"YES": "Yes", "NO": "No", "UNKNOWN": "Unknown"}
def canon(x):
    u = norm(x)
    return yn_map.get(u, u)

# ================================================================
# TEST 1 — Per-class accuracy (Z3 vs Qwen vs Gold)
# Mục đích: thấy ngay Z3 underfit class "No"
# ================================================================
print("\n" + "="*65)
print("  TEST 1: Per-class accuracy (Z3 vs Qwen)")
print("="*65)
from collections import defaultdict
z3_by_class  = defaultdict(lambda: [0,0])   # {class: [correct, total]}
qw_by_class  = defaultdict(lambda: [0,0])
for r in records:
    g = canon(r["gold"])
    z_correct = int(canon(r["z3_answer"]) == g)
    q_correct = int(canon(r["qwen_answer"]) == g)
    z3_by_class[g][0] += z_correct; z3_by_class[g][1] += 1
    qw_by_class[g][0] += q_correct; qw_by_class[g][1] += 1

print(f"\n  {'Class':<10} {'Z3 acc':>10} {'Qwen acc':>10} {'Count':>8}")
print(f"  {'-'*42}")
for cls in sorted(z3_by_class.keys(), key=lambda x: -z3_by_class[x][1]):
    zc, zt = z3_by_class[cls]
    qc, qt = qw_by_class[cls]
    z_acc = f"{zc/zt:.1%}" if zt else "n/a"
    q_acc = f"{qc/qt:.1%}" if qt else "n/a"
    flag = " ← Z3 underfit" if cls in ("No","NO") and zt and zc/zt < 0.5 else ""
    print(f"  {cls:<10} {z_acc:>10} {q_acc:>10} {zt:>8}{flag}")

# ================================================================
# TEST 2 — BoN vote spread: "Qwen chắc chắn hay memorize?"
# Mục đích: nếu spread=1 (5/5 vote cùng) quá cao → nghi memorize
# ================================================================
print("\n" + "="*65)
print("  TEST 2: BoN vote spread distribution (Qwen)")
print("="*65)
if "qwen_vote_spread" in records[0]:
    spread_cnt = Counter(r["qwen_vote_spread"] for r in records)
    total = len(records)
    print(f"\n  Spread=1 (5/5 vote cùng): {spread_cnt.get(1,0)/total:.1%}  ← nếu >70%: nghi memorize")
    print(f"  Spread=2               : {spread_cnt.get(2,0)/total:.1%}")
    print(f"  Spread=3               : {spread_cnt.get(3,0)/total:.1%}")
    print(f"  Spread≥4               : {sum(v for k,v in spread_cnt.items() if k>=4)/total:.1%}")

    # Accuracy breakdown by spread
    print(f"\n  Accuracy by spread (qwen):")
    spread_acc = defaultdict(lambda: [0,0])
    for r in records:
        s = r["qwen_vote_spread"]
        spread_acc[s][0] += r.get("qwen_correct",0)
        spread_acc[s][1] += 1
    for s in sorted(spread_acc.keys()):
        c, t = spread_acc[s]
        bar = "█"*int(30*c/t) if t else ""
        print(f"  spread={s}: {c/t:.1%} ({c}/{t}) {bar}")

    # Calibration: high confidence + wrong
    high_conf_wrong = [r for r in records
                       if r.get("qwen_sc_conf",0) >= 0.8
                       and r.get("qwen_correct",0) == 0]
    print(f"\n  High-conf (≥0.8) but WRONG: {len(high_conf_wrong)}/{len(records)} ({len(high_conf_wrong)/len(records):.1%})")
    print(f"  ← Nếu >5%: Qwen chắc chắn sai = overfit signal")
    if high_conf_wrong[:3]:
        print(f"\n  Ví dụ 3 câu high-conf-wrong:")
        for r in high_conf_wrong[:3]:
            print(f"    gold={r['gold']}  qwen={r['qwen_answer']}  conf={r.get('qwen_sc_conf',0):.2f}")
            print(f"    q_type={r['q_type']}  z3={r['z3_answer']}  z3_correct={r.get('z3_correct',0)}")
else:
    print("  (chỉ có trong v13.6 — cần qwen_sc_conf)")

# ================================================================
# TEST 3 — Calibrator k-fold (5-fold)
# Mục đích: kiểm tra 73.8% val có ổn định không hay chỉ may mắn
# ================================================================
print("\n" + "="*65)
print("  TEST 3: Calibrator k-fold (5-fold, từ feature cache)")
print("="*65)

FEATURES = ["z3_conf","z3_grounded","z3_definite","z3_spread","z3_qwen_agree",
            "q_type_mc","n_idx_premises","has_free_var","is_converse",
            "n_quantifiers","stmt_len"]
# Thêm v13.6 features nếu có
extra = ["qwen_sc_conf","qwen_vote_spread","qwen_n_cited","z3_qwen_jaccard"]
FEATURES += [f for f in extra if f in records[0]]
print(f"  Features: {len(FEATURES)} = {FEATURES}")

X = np.array([[r[f] for f in FEATURES] for r in records], dtype=float)
y = np.array([r.get("z3_correct",0) for r in records], dtype=int)
gold_ans = [canon(r["gold"]) for r in records]
z3_ans   = [canon(r["z3_answer"]) for r in records]
qw_ans   = [canon(r["qwen_answer"]) for r in records]

try:
    import lightgbm as lgb
    clf_fn = lambda: lgb.LGBMClassifier(n_estimators=300, num_leaves=15,
                      learning_rate=0.04, min_child_samples=10,
                      random_state=42, verbose=-1)
    backend = "lightgbm"
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    clf_fn = lambda: GradientBoostingClassifier(n_estimators=300, max_depth=3,
                      learning_rate=0.04, random_state=42)
    backend = "sklearn-gbm"

import random
random.seed(42)
# Sample-level k-fold (same as pipeline: split by sample_id)
sample_ids = sorted(set(r["sample_id"] for r in records))
random.shuffle(sample_ids)
K = 5
fold_sz = len(sample_ids) // K
fold_accs = []

print(f"\n  backend={backend}, K=5, n_samples={len(sample_ids)}")
for k in range(K):
    val_ids   = set(sample_ids[k*fold_sz : (k+1)*fold_sz])
    train_ids = set(sample_ids) - val_ids
    tr_idx = [i for i,r in enumerate(records) if r["sample_id"] in train_ids]
    va_idx = [i for i,r in enumerate(records) if r["sample_id"] in val_ids]
    clf = clf_fn()
    clf.fit(X[tr_idx], y[tr_idx])
    p = clf.predict_proba(X[va_idx])[:,1]
    # Tune tau on train
    best_tau, best_acc = 0.5, 0.0
    for tau in [x/100 for x in range(20,90,5)]:
        c = sum(1 for j,vi in enumerate(va_idx)
                if (z3_ans[vi] if (records[vi]["z3_definite"] and p[j]>=tau) else qw_ans[vi])
                == gold_ans[vi])
        a = c/len(va_idx)
        if a > best_acc: best_acc,best_tau = a,tau
    fold_accs.append(best_acc)
    print(f"  fold {k+1}: val_acc={best_acc:.1%} (tau={best_tau:.2f}) n_val={len(va_idx)}")

mean_acc = np.mean(fold_accs)
std_acc  = np.std(fold_accs)
print(f"\n  k-fold result: {mean_acc:.1%} ± {std_acc:.1%}")
print(f"  single-split val (before): 73.8%")
if std_acc > 0.03:
    print(f"  ← std cao ({std_acc:.1%}) = số không ổn định → confirm overfit")
elif mean_acc > 0.72:
    print(f"  ← stable! Calibrator not significantly overfit")
else:
    print(f"  ← mean thấp hơn single split → single split đã lucky")

# ================================================================
# TEST 4 — "No" class deep dive (Z3 underfit confirmation)
# ================================================================
print("\n" + "="*65)
print("  TEST 4: Z3 class-No deep dive")
print("="*65)
no_records = [r for r in records if canon(r["gold"]) == "No"]
if no_records:
    z3_def_on_no = [r for r in no_records if r["z3_definite"]]
    z3_right_no  = [r for r in z3_def_on_no if canon(r["z3_answer"])=="No"]
    z3_wrong_no  = [r for r in z3_def_on_no if canon(r["z3_answer"])!="No"]
    qw_right_no  = [r for r in no_records if r.get("qwen_correct",0)]
    print(f"\n  Total 'No' gold: {len(no_records)}")
    print(f"  Z3 definite on No gold: {len(z3_def_on_no)} ({len(z3_def_on_no)/len(no_records):.0%})")
    if z3_def_on_no:
        print(f"  Z3 correct (answer=No): {len(z3_right_no)} ({len(z3_right_no)/len(z3_def_on_no):.1%})")
        print(f"  Z3 wrong  (answer≠No): {len(z3_wrong_no)} ({len(z3_wrong_no)/len(z3_def_on_no):.1%})")
        wrong_ans = Counter(canon(r["z3_answer"]) for r in z3_wrong_no)
        print(f"  Z3 wrong predictions: {dict(wrong_ans)}")
        print(f"  ← Nếu Z3 chủ yếu nói 'YES' khi gold='No' → Z3 underfit class No")
    print(f"  Qwen correct on No gold: {len(qw_right_no)} ({len(qw_right_no)/len(no_records):.1%})")

# ================================================================
# TEST 5 — v14 LoRA: đọc trainer_state để kiểm tra train vs eval loss
# ================================================================
print("\n" + "="*65)
print("  TEST 5: v14 LoRA training curve (trainer_state.json)")
print("="*65)
TRAINER_CANDIDATES = [
    "/kaggle/input/notebooks/nguyenminhtric/v14-fine-tune/cot_lora_ckpt/checkpoint-126/trainer_state.json",
    "/kaggle/working/cot_lora_ckpt/checkpoint-126/trainer_state.json",
    "/kaggle/working/qwen3_cot_lora_v14_v4/trainer_state.json",
]
state = None
for p in TRAINER_CANDIDATES:
    if os.path.exists(p):
        state = json.load(open(p)); print(f"  Found: {p}"); break
if state is None:
    print("  trainer_state.json không tìm thấy — bỏ qua test này.")
    print("  Để chạy: đảm bảo path /kaggle/input/notebooks/.../trainer_state.json đúng")
else:
    log_history = state.get("log_history", [])
    train_logs = [x for x in log_history if "loss" in x and "eval_loss" not in x]
    eval_logs  = [x for x in log_history if "eval_loss" in x]
    print(f"\n  Total log entries: train={len(train_logs)} eval={len(eval_logs)}")
    if train_logs:
        print(f"\n  Training loss progression:")
        for entry in train_logs:
            step = entry.get("step","?"); loss = entry.get("loss","?")
            print(f"    step {step:>4}: train_loss = {loss}")
    if eval_logs:
        print(f"\n  Eval loss progression:")
        for entry in eval_logs:
            step = entry.get("step","?"); eloss = entry.get("eval_loss","?")
            print(f"    step {step:>4}: eval_loss = {eloss:.4f}" if isinstance(eloss,float) else f"    step {step:>4}: eval_loss = {eloss}")
        if len(eval_logs) >= 2:
            first_eval = eval_logs[0].get("eval_loss", 0)
            last_eval  = eval_logs[-1].get("eval_loss", 0)
            if isinstance(first_eval, float) and isinstance(last_eval, float):
                if last_eval > first_eval * 1.05:
                    print(f"\n  ← OVERFIT SIGNAL: eval_loss TĂNG ({first_eval:.4f} → {last_eval:.4f})")
                else:
                    print(f"\n  ← OK: eval_loss giảm/ổn định ({first_eval:.4f} → {last_eval:.4f})")

print("\n" + "="*65)
print("  DIAGNOSTIC SUMMARY")
print("="*65)
print("""
  Test 1 (per-class):   Z3 underfit 'No' đã xảy ra (có số)?
  Test 2 (BoN spread):  Qwen spread=1 >70%? High-conf-wrong >5%?
  Test 3 (k-fold):      Calibrator mean ± std so với 73.8%?
  Test 4 (No class):    Z3 nói 'Yes' bao nhiêu lần khi gold='No'?
  Test 5 (LoRA curve):  eval_loss tăng sau epoch nào?
  
  → Nếu Test 1+4 confirm Z3 sai trên 'No': underfit đã xảy ra
  → Nếu Test 2 spread=1 >75%: memorize signal (cần external test confirm)
  → Nếu Test 3 std >3%: calibrator số không tin được
  → Nếu Test 5 eval_loss tăng: LoRA overfit lúc train
""")
